# Generate a dataset for fine-tuning SBERT 

### Step 1: Generate weak labels 
Label subset of the data using rule-based regexing, if keywords are present, then assign the label. 

### Step 2: Prepare the data as matching pairs
Build positive pairs from the weak labels, by pairing comments according to the similarity of the labels assigned. Use soft-confidence scores to account for the fact that the labelling in step 1 is weak. 

In [1]:
import re
import random

import pickle

from collections import Counter
from itertools import combinations 

from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sentence_transformers import InputExample

import sys
sys.path.append('../pipeline')
from nlp_tasks import NLP_Tasks
from comments_saver import CommentsSaver

/opt/miniconda3/envs/nlp_env_analysis/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Read the remote database of comments

In [2]:
cs = CommentsSaver(env='dev')
df = cs.read_all()
print('df shape:', df.shape)

Connecting to the ai4ci-db-dev database...
Successfully connected to ai4ci-db-dev.
df shape: (30994, 11)


In [3]:
# remove duplicate comments from the training set
df = df.drop_duplicates(subset=['comment_text']).reset_index(drop=True)
print('df training set shape after removing duplicates:', df.shape)

df training set shape after removing duplicates: (29117, 11)


In [4]:
df.head()

,id,council,comment_id,application_id,address,stance,date,comment_text,add_date,lat,lon
0,108714,Ealing,223084PANDF_2,223084PANDF,55 azalea court azalea close hanwell w73qa w73qa,Objects,2022-10-25,"As a resident of Azalea Court, I strongly obje...",2025-05-01,NaN,NaN
1,100341,Ealing,201888FUL_145,201888FUL,5 Green Drive Southall UB1 3AY,Objects,2020-07-14,"Traffic, pollution and over-population is alre...",2025-05-01,51.508850,-0.366200
2,108719,Ealing,223084PANDF_6,223084PANDF,70 Eccleston Road Ealing W13 0rl W13 0rl,Objects,2022-08-18,On going over development is changing the plea...,2025-05-01,51.511287,-0.329087
3,108715,Ealing,223084PANDF_3,223084PANDF,19 Azalea Close Ealing London W7 3QA W7 3QA,Objects,2022-10-24,- No more than half the 10 additional cars or ...,2025-05-01,51.510060,-0.332280
4,108718,Ealing,223084PANDF_5,223084PANDF,36 Azalea Close Hanwell London W7 3QA W7 3QA,Objects,2022-08-18,This is a petition from residents in no's 3/6 ...,2025-05-01,51.510060,-0.332280


### Split a small fraction of the dataset to label 
Stratify the split according to 'application_id', 'stance' and 'council'

In [5]:
# Group by 'application_id' to create a stratified train test split across application ids. 
# Just splittling a tiny fraction (1%) for the sake of the example
gss = GroupShuffleSplit(n_splits=1, train_size=0.1, random_state=42)

# Get the application_id groups
groups = df['application_id']

# Split the indices
train_inds, test_inds = next(gss.split(df, groups=groups))

# Create train and test DataFrames
df_train = df.iloc[train_inds].reset_index(drop=True)

In [6]:
print('df training set shape:', df_train.shape)

df training set shape: (2650, 11)


### Pre-process the data 

In [7]:
nlp = NLP_Tasks()

# remove place names, numbers and non-ASCII characters
# df = nlp.remove_place_names(df=df, column='comment_text')
# df = nlp.remove_numbers(df=df, column='cleaned_comment_text')

# remove non-ASCII characters
df_train = nlp.remove_non_ascii(df=df_train, column='comment_text')

# split text on newlines
df_train = nlp.split_text_on_newline(df=df_train, column='comment_text', filter_empty=True, filter_short=True, min_length=5)
print('df training set shape:', df_train.shape)

Device set to use mps:0


df training set shape: (16093, 11)


### Define the terms for the regex matching - heuristic 

In [8]:
issue_topic_map = {
    "Out of character with area": r"\b(character|incongruous|appearance|architectural|visual|aesthetic|look|design|style|context|blend|beauty|charm|out of keeping)\b",
    "Scale of development": r"\b(scale|bulk|height|dominance|overbearing|oversized|mass|tall|overdevelop|over develop)\b",
    "Density of development": r"\b(density|dense|overdevelopment|crammed|crowded|too many units|congested|overcrowding)\b",
    "Impact on heritage": r"\b(heritage|listed|conservation|historic|preserv)\b",
    "Housing mix wrong": r"\b(mix|family|bedroom|flats only|unit type|tenure|local housing needs|social housing|council housing)\b",
    "Not affordable": r"\b(affordable|affordability|social housing|council housing|luxury|market rate)\b",
    "Impact on local amenities": r"\b(amenity|impact on|local high street|commercial units)\b",
    "Impact on social infrastructure": r"\b(school|no public benefits|gp|clinic|healthcare|park|playground|community service|childcare|local resources)\b",
    "Impact on transport infrastructure": r"\b(traffic|highway|transport|parking|car|bus|congestion|accessibility)\b",
    "Impact on garbage infrastructure": r"\b(waste|garbage|bin|rubbish|trash|collection|disposal|sanitation|litter|dump|overflow|vermin|rodent)\b",
    "Impact on transport safety": r"\b(safety|dangerous|accident|pedestrian|footpath|crossing|collision|vehicle hazard)\b",
    "Increased air pollution": r"\b(air quality|pollution|dust|emission|fumes|vehicle exhaust)\b",
    "Increased noise pollution": r"\b(noise|loud|disturbance|sound|construction noise|disruption)\b",
    "Loss of green space": r"\b(green space|trees|garden|open space|vegetation|greenery|nature|green spaces|outdoor space)\b",
    "Damage to natural environment": r"\b(environment|wildlife|biodiversity|habitat|nature|ecology)\b",
    "Loss of parking": r"\b(parking|car space|vehicle storage|no place to park|permit|number of vehicles)\b",
    "Impact on water infrastructure": r"\b(flood|drainage|surface water|sewer|runoff|waterlog|waterpipe|water pipe|water flow|water board)\b",
    "Inadequate public consultation": r"\b(consultation|not informed|residents ignored|no say|ballot|engagement|misled)\b",
    "Loss of privacy": r"\b(privacy|overlook|see into|direct view|window intrusion|overlooking)\b",
    "Loss of light": r"\b(light|shadow|sunlight|daylight|block sun|loss of light|overshadow)\b",
    "Impact on visual amenity": r"\b(visual|eyesore|ugly|disturbance|view ruined|inappropriate look)\b",
    "Disabled access": r"\b(disabled|wheelchair|mobility|accessible|step-free|inclusive design)\b",
    "Archaeology": r"\b(archaeolog|historic site|ancient|dig|relic)\b",
    "Right of way": r"\b(right of way|footpath|access route|public access|pathway blocked)\b",
    "Contractual obligations": r"\b(covenant|agreement|legal obligation|contract|binding terms)\b",
    "Applicants character": r"\b(developer trust|past record|reputation|history|untrustworthy|previous projects)\b",
    "Loss of view": r"\b(view|scenic|vista|lookout|blocked view)\b",
    "Loss of property value": r"\b(value|property price|devaluation|resale)\b",
    "Impact on trade": r"\b(trade|business|shop|retail|commercial impact|livelihood)\b",
    "Volume of objections": r"\b(objections|petition|local opposition|many comments|campaign)\b",
    "Construction disturbance": r"\b(building work|dust|disruption|noise from works|timeline)\b",
    "Lack of maintenance": r"\b(maintenance|neglect|disrepair|upkeep|deterioration)\b",
    "Neighbourhood disputes": r"\b(neighbour|dispute|argument|row|resident disagreement)\b",
    "Not meeting environmental standards": r"\b(environmental standard|carbon|eco|sustainable|green standard|BREEAM|passivhaus|climate)\b",
    "Concerns about precarity": r"\b(precarity|uncertain future|unstable|temporary housing|no guarantee)\b",
    "Misleading application": r"\b(misleading|inaccurate|dishonest|bait and switch|changed plans|false information|lack measurements)\b",
    "Replacement accommodation not suitable": r"\b(replacement housing|unsuitable|not like for like|no garden|flat instead of house)\b",
    "Residents forced to leave area": r"\b(evict|move away|displaced|no return|gentrification|moved out)\b",
    "Developer previously built below standard dwellings": r"\b(poor quality|shoddy work|track record|bad build|low standard)\b",
    "Increase in antisocial behaviour": r"\b(antisocial|crime|drugs|violence|unruly|unsafe)\b",
    "Will improve safety": r"\b(safety improvement|better lighting|secure|surveillance|safe area)\b",
    "Area needs regeneration": r"\b(regeneration|revitalise|modernise|redevelop|run-down)\b",
    "Better replacement accomodation": r"\b(better housing|modern homes|improved living|higher quality|new build benefit)\b",
    "Area needs more housing": r"\b(housing need|shortage|waiting list|supply|more homes)\b",
    "Meets standards": r"\b(standard met|compliant|satisfies code|approved|safe design|meets regulation)\b",
    "In character with local area": r"\b(is in keeping|matches area|context appropriate|fits in|neighbourhood style)\b",
    "Private only spaces": r"\b(private|exclusive|residents only|no public access|gated|segregated)\b",
    "Lack of space for social distancing": r"\b(crowded|too close|no distancing|compact|post-COVID)\b",
    "Accomodation unsuitable for vulnerable people": r"\b(vulnerable|elderly|disabled|high rise harm|inaccessible)\b",
    "Boundary disputes": r"\b(boundary wall|dividing wall|property line|encroachment|land dispute)\b",
}


### Apply the regex matching algorithm 

In [9]:
def assign_topics(comment):
    matched_topics = []
    for topic, pattern in issue_topic_map.items():
        if re.search(pattern, comment, flags=re.IGNORECASE):
            matched_topics.append(topic)
    return matched_topics if matched_topics else ["No match"]

# Apply to DataFrame
df_train['assigned_topics'] = df_train['comment_text'].apply(assign_topics)

In [10]:
# Save the DataFrame with assigned topics to a CSV file
df_train[['stance', 'comment_text', 'assigned_topics']].to_csv('../outputs/SBERT_pairs/df_train.csv', index=False)

### Can also use stemming and fuzzy matching 
Stemming is when you match the stem word, and don't about the case. Fuzzy matching is when you allow for variation, including stemming and similar words/terms. 
I've found that this over macthes the terms, and everyhting gets assigned every topic label. 

In [11]:
# import re
# from nltk.stem import PorterStemmer
# from fuzzysearch import find_near_matches

# ps = PorterStemmer()

# # Preprocess keywords into stemmed versions
# stemmed_issue_topic_map = {
#     topic: [ps.stem(word) for word in re.findall(r'\w+', pattern)]
#     for topic, pattern in issue_topic_map.items()
# }

# def assign_topics(comment):
#     matched_topics = []
#     comment_lower = comment.lower()
#     comment_words = re.findall(r'\w+', comment_lower)
#     stemmed_comment_words = [ps.stem(word) for word in comment_words]

#     for topic, keywords in stemmed_issue_topic_map.items():
#         for stemmed_kw in keywords:
#             for word in comment_words:
#                 if ps.stem(word) == stemmed_kw:
#                     matched_topics.append(topic)
#                     break
#                 # Fuzzy match with max 1 edit distance
#                 if find_near_matches(stemmed_kw, word, max_l_dist=1):
#                     matched_topics.append(topic)
#                     break
#             else:
#                 continue
#             break

#     return list(set(matched_topics)) if matched_topics else ["No match"]

# # Apply to DataFrame
# df_train['assigned_topics'] = df_train['comment_text'].apply(assign_topics)


In [12]:
# Count rows where 'assigned_topics' is exactly ['No match']
no_match_count = df_train['assigned_topics'].apply(lambda x: x == ['No match']).sum()
total_count = df_train.shape[0]
no_match_percent = (no_match_count / total_count) * 100

print(f"Percentage of 'No match' comments: {no_match_percent:.2f}%")

Percentage of 'No match' comments: 36.27%


In [13]:
# Flatten all topic lists into a single list
all_topics = [topic for sublist in df_train['assigned_topics'] for topic in sublist if topic != "No match"]

# Count occurrences of each topic
topic_counter = Counter(all_topics)

# Calculate percentage of rows associated with each topic
topic_percentages = {topic: (count / total_count) * 100 for topic, count in topic_counter.items()}

# Display results
print("Percentage of rows associated with each topic:")
for topic, percent in sorted(topic_percentages.items(), key=lambda x: -x[1]):
    print(f"{topic}: {percent:.2f}%")


Percentage of rows associated with each topic:
Out of character with area: 14.68%
Impact on transport infrastructure: 12.58%
Loss of green space: 12.08%
Loss of parking: 8.79%
Scale of development: 7.59%
Impact on local amenities: 6.79%
Impact on heritage: 6.78%
Loss of light: 6.72%
Loss of privacy: 5.58%
Density of development: 5.38%
Increased noise pollution: 5.18%
Impact on social infrastructure: 4.44%
Damage to natural environment: 3.78%
Housing mix wrong: 3.56%
Increased air pollution: 2.90%
Impact on visual amenity: 2.57%
Impact on transport safety: 2.53%
Impact on garbage infrastructure: 2.16%
Construction disturbance: 1.85%
Loss of view: 1.75%
Volume of objections: 1.72%
Not meeting environmental standards: 1.60%
Not affordable: 1.21%
Private only spaces: 1.17%
Meets standards: 1.08%
Neighbourhood disputes: 1.03%
Loss of property value: 1.00%
Impact on trade: 0.98%
Accomodation unsuitable for vulnerable people: 0.78%
Impact on water infrastructure: 0.75%
Lack of space for socia

In [14]:
# return count of items with mutliple matched topics
multiple_matches = df_train[df_train['assigned_topics'].apply(lambda x: len(x) > 1)]
print(f"Number of comments with multiple matched topics: {len(multiple_matches)}")

Number of comments with multiple matched topics: 5935


In [15]:
df_train_no_match = df_train[df_train['assigned_topics'].apply(lambda x: x == ['No match'])].reset_index(drop=True)
df_train_no_match[['stance', 'comment_text', 'assigned_topics']].to_csv('../outputs/SBERT_pairs/df_train_no_match.csv', index=False)

### Create positive and negative pairs for contrastive loss 
To reduce the noise in my labels I filter out the comments with 'No match' or no assigned topics. 

In [16]:
# Filter out rows with no valid topics
df_train = df_train[df_train['assigned_topics'].apply(lambda x: 'No match' not in x and len(x) > 0)]

Use soft labels for the pairs - this accounts for the fact that the data was quite weakly labelled using the heuristic approach of regex matching. 

### Soft-labelled positive pairs 
* all topics shared -> label=1
* all but one topic shared -> label=0.75
* all but two topics shared -> label=0.5

Then generate the same number of negative pairs - which is useful for contrastive loss training. I just sample a random set of these - as could potentially generate an enormous number of these, since n choose 2 etc. 
### negative pairs 
* 0 shared topics -> label=0

In [17]:
# Build all valid comment-topic pairs
comments = df_train[['comment_text', 'assigned_topics']].values.tolist()
positive_pairs = []

def compute_soft_label(topics1, topics2):
    set1, set2 = set(topics1), set(topics2)
    if set1 == set2:
        return 1.0
    shared = set1.intersection(set2)
    total = max(len(set1.union(set2)), 1)
    diff_count = total - len(shared)
    
    if len(shared) == 0:
        return None  # will be handled as negative
    elif diff_count == 1:
        return 0.75
    elif diff_count == 2:
        return 0.5
    else:
        return None  # skip low-similarity cases

# Generate positive pairs with refined labels
for (text1, topics1), (text2, topics2) in combinations(comments, 2):
    score = compute_soft_label(topics1, topics2)
    if score is not None:
        positive_pairs.append(InputExample(texts=[text1, text2], label=score))

print(f"Generated {len(positive_pairs)} refined positive pairs.")

# Generate balanced hard negatives
negative_pairs = []
while len(negative_pairs) < len(positive_pairs):
    text1, topics1 = random.choice(comments)
    text2, topics2 = random.choice(comments)
    if text1 != text2 and set(topics1).isdisjoint(set(topics2)):
        negative_pairs.append(InputExample(texts=[text1, text2], label=0.0))

print(f"Generated {len(negative_pairs)} negative pairs.")

# Combine for training
train_examples = positive_pairs + negative_pairs

Generated 4454943 refined positive pairs.
Generated 4454943 negative pairs.


In [18]:
for i, ex in enumerate(train_examples[:5]):
    print(f"Pair {i+1} - Label: {ex.label}")
    print("Text 1:", ex.texts[0][:150], "...")
    print("Text 2:", ex.texts[1][:150], "...")
    print("-" * 80)


Pair 1 - Label: 0.5
Text 1: - Excessive height: it will have a negative impact on the area, overshadowing and overlooking surrounding properties. The South Acton Estate was demol ...
Text 2: The proposed 23 story building is too high for the area and runs the risk of attracting other similarly tall buildings. One look at North Acton shows  ...
--------------------------------------------------------------------------------
Pair 2 - Label: 0.5
Text 1: - Excessive height: it will have a negative impact on the area, overshadowing and overlooking surrounding properties. The South Acton Estate was demol ...
Text 2: 1 - The proposal is out of keeping with the area to the South (Southfield) and also taller than most of the buildings in the South Acton area. The Cou ...
--------------------------------------------------------------------------------
Pair 3 - Label: 0.5
Text 1: - Excessive height: it will have a negative impact on the area, overshadowing and overlooking surrounding properties. 

In [19]:
# Save train_examples to a file
with open('../outputs/SBERT_pairs/train_examples.pkl', 'wb') as f:
    pickle.dump(train_examples, f)